# 01_nlp_introduction: Raw Text Ingestion and Normalization Pipeline

This notebook demonstrates the standard NLP ingestion and preprocessing pipeline. It fetches real-world text, cleans it by removing HTML tags and regex-defined noise, maps tokens to vocabulary indices, and compares NFC vs NFD Unicode normalizations to prevent tokenizer misalignment.

In [1]:
import requests
from bs4 import BeautifulSoup

# Fetch raw text from Wikipedia's NLP page
url = "https://en.wikipedia.org/wiki/Natural_language_processing"
headers = {'User-Agent': 'Mozilla/5.0'}
resp = requests.get(url, headers=headers)
soup = BeautifulSoup(resp.content, "html.parser")

# Select paragraphs and slice the first substantial block
paragraphs = [p.get_text().strip() for p in soup.find_all("p") if len(p.get_text().strip()) > 80]
raw_paragraph = paragraphs[1]

print("Raw Ingested Paragraph snippet:")
print(raw_paragraph[:150], "...")
assert len(raw_paragraph) > 0, "Ingestion failed!"

Raw Ingested Paragraph snippet:
Major processing tasks in an NLP system include: speech recognition, text classification, natural language understanding, and natural language generat ...


### Output Explanation: Raw Ingestion
- **Scraped Content:** We successfully retrieved raw, unformatted text from Wikipedia. The text contains capitalizations, punctuation marks, and structural words that need to be normalized.
- **Context:** This represents the raw input layer in a production pipeline prior to syntactic extraction.

In [2]:
import re
import nltk
nltk.download('punkt', quiet=True)
from nltk.tokenize import word_tokenize

# Define noise patterns (URLs, numbers, punctuation)
url_pattern = r"https?://\S+|www\.\S+"
non_alpha_pattern = r"[^a-zA-Z\s]"

# Clean the text
cleaned_text = re.sub(url_pattern, "", raw_paragraph)
cleaned_text = re.sub(non_alpha_pattern, "", cleaned_text).lower()
cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

# Tokenize
tokens = word_tokenize(cleaned_text)[:15] # Take a subset of 15 tokens

print("Cleaned Text snippet:")
print(cleaned_text[:120], "...")
print("\nFirst 15 Word Tokens:")
print(tokens)

# Assertions to verify cleaning correctness
assert cleaned_text.islower(), "Text is not completely lowercased!"
assert not re.search(url_pattern, cleaned_text), "URL patterns remain!"
assert len(tokens) == 15, "Tokenization slice mismatch!"

Cleaned Text snippet:
major processing tasks in an nlp system include speech recognition text classification natural language understanding an ...

First 15 Word Tokens:
['major', 'processing', 'tasks', 'in', 'an', 'nlp', 'system', 'include', 'speech', 'recognition', 'text', 'classification', 'natural', 'language', 'understanding']


### Output Explanation: Preprocessing and Tokenization
- **Regex Normalization:** Capitalization is unified to lowercase using `.lower()`. Punctuation (such as commas and periods) is stripped via `re.sub`. This ensures words like `"System"` and `"system"` map to the same vocabulary dimension.
- **Whitespace Consolidation:** Multiple space characters are collapsed to single spaces.
- **Word Tokenization:** The cleaned string is segmented into word-level tokens, matching the standard token representation layer.

In [3]:
import unicodedata

# Define an accented string containing é (e with acute accent)
accented_str_nfc = "café"  # NFC composed
accented_str_nfd = unicodedata.normalize('NFD', accented_str_nfc)  # NFD decomposed

print(f"NFC String: '{accented_str_nfc}' | Length: {len(accented_str_nfc)} | Code points: {[ord(c) for c in accented_str_nfc]}")
print(f"NFD String: '{accented_str_nfd}' | Length: {len(accented_str_nfd)} | Code points: {[ord(c) for c in accented_str_nfd]}")

# Show matching issues
print(f"Direct equality check (NFC == NFD): {accented_str_nfc == accented_str_nfd}")

# Regex match test
regex_pattern = r"^caf\u00e9$"  # Matches NFC é
match_nfc = re.match(regex_pattern, accented_str_nfc)
match_nfd = re.match(regex_pattern, accented_str_nfd)
print(f"Regex matches NFC: {bool(match_nfc)} | Regex matches NFD: {bool(match_nfd)}")

# Assertions
assert len(accented_str_nfc) == 4, "NFC length mismatch!"
assert len(accented_str_nfd) == 5, "NFD length mismatch!" # e + combining acute accent
assert accented_str_nfc != accented_str_nfd, "Unnormalized strings should not match!"

NFC String: 'café' | Length: 4 | Code points: [99, 97, 102, 233]
NFD String: 'café' | Length: 5 | Code points: [99, 97, 102, 101, 769]
Direct equality check (NFC == NFD): False
Regex matches NFC: True | Regex matches NFD: False


### Output Explanation: Unicode Normalization
- **Decomposition (NFD):** Decomposes the single code point `é` (`\u00e9`) into two characters: the base character `e` (`\u0065`) and the combining acute accent character `´` (`\u0301`). This increases the string length of `"café"` from 4 to 5.
- **Production Risk:** Naive string matches or regex patterns fail when comparing NFC and NFD encodings, even though they render identically. This illustrates why preprocessing must include normalization (like `unicodedata.normalize('NFC', text)`) to prevent vocabulary alignment failures.